# Simple base-rate merged results

Load multi-model results from downloaded Kaggle Benchmarks runs, or from a local merged CSV.

Each row has **`score`** (`true`/`false`): whether the parsed answer matches normative **P(C|T)**. **`path_c_confusion`** flags answers matching **P(T|C)** (the inverse-conditional lure).

**Kaggle (all evaluated models):** after `kaggle auth login`:

```powershell
python -m kaggle benchmarks tasks download simple-rate-normative-accuracy `
  -o data/kaggle_runs/simple-rate-normative-accuracy
```

Or: `python scripts/export_simple_rate_kaggle_results.py --download`

Set `LOAD_FROM_KAGGLE = True` in the next cell (default).

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "simple").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.kaggle_runs import (
    DEFAULT_SIMPLE_RATE_TASK_SLUG,
    download_task_runs,
    find_run_json_files,
    merged_simple_results_from_kaggle_runs,
)

LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = False  # True to refresh via Kaggle CLI before loading
KAGGLE_TASK_SLUG = DEFAULT_SIMPLE_RATE_TASK_SLUG
KAGGLE_RUNS_DIR = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
BENCHMARK_CSV = ROOT / "data" / "simple" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "simple"

if LOAD_FROM_KAGGLE:
    if DOWNLOAD_KAGGLE_RUNS:
        download_task_runs(KAGGLE_TASK_SLUG, KAGGLE_RUNS_DIR)
    if not KAGGLE_RUNS_DIR.is_dir():
        raise FileNotFoundError(
            f"Download directory not found: {KAGGLE_RUNS_DIR}\n"
            f"Run: python -m kaggle benchmarks tasks download {KAGGLE_TASK_SLUG} "
            f"-o {KAGGLE_RUNS_DIR}"
        )
    run_files = find_run_json_files(KAGGLE_RUNS_DIR)
    merged_rows = merged_simple_results_from_kaggle_runs(
        KAGGLE_RUNS_DIR,
        benchmark_path=BENCHMARK_CSV,
    )
    df = pd.DataFrame(merged_rows)
    data_source = f"Kaggle runs ({len(run_files)} *.run.json under {KAGGLE_RUNS_DIR})"
else:
    merged_candidates = sorted(
        MERGED_DIR.glob("simple_merged_results*.csv"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not merged_candidates:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/simple-benchmark.ipynb."
        )
    MERGED_CSV = merged_candidates[0]
    df = pd.read_csv(MERGED_CSV)
    data_source = str(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
else:
    raise KeyError("Merged data must include 'score'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")
else:
    df["parseable_bool"] = True

if "path_c_confusion" in df.columns:
    df["path_c_confusion_bool"] = (
        df["path_c_confusion"].astype(str).str.lower().eq("true")
    )
else:
    df["path_c_confusion_bool"] = False

df["score_true"] = df["score_value"].astype(bool)

VARIANT_ORDER = ["open_probs", "mc_numeric_probs"]

print("Source:", data_source)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
print(
    "Normative pass:",
    int(df["score_true"].sum()),
    "/",
    len(df),
    "| P(T|C) confusion:",
    int(df["path_c_confusion_bool"].sum()),
    "/",
    len(df),
)
df.head()

Source: Kaggle runs (12 *.run.json under c:\src2\sceptical-llms\data\kaggle_runs\simple-rate-normative-accuracy)
Rows: 108
Models: ['anthropic/claude-opus-4-1@20250805', 'anthropic/claude-opus-4-8@default', 'anthropic/claude-sonnet-4@20250514', 'google/gemini-3-flash-preview', 'google/gemini-3.5-flash', 'openai/gpt-5.5-2026-04-23']
Vignettes: 9
Normative pass: 30 / 108 | P(T|C) confusion: 33 / 108


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_choice,parsed_confidence,scoring_type,parseable,score,path_c_confusion,score_value,parseable_bool,path_c_confusion_bool,score_true
0,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc,true,mc_numeric_probs,You are a statistical consultant. Your task is...,true,well_posed,...,,,mc_numeric,false,false,false,0,False,False,False
1,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc,true,mc_numeric_probs,You are a statistical consultant. Your task is...,true,well_posed,...,A,,mc_numeric,true,true,false,1,True,False,True
2,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc,true,mc_numeric_probs,You are a statistical consultant. Your task is...,true,well_posed,...,,,mc_numeric,false,false,false,0,False,False,False
3,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc,true,mc_numeric_probs,You are a statistical consultant. Your task is...,true,well_posed,...,A,,mc_numeric,true,true,false,1,True,False,True
4,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc,true,mc_numeric_probs,You are a statistical consultant. Your task is...,true,well_posed,...,A,,mc_numeric,true,true,false,1,True,False,True


In [2]:
def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, normative score, and P(T|C) confusion by group."""
    work = df.copy()
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "path_c_confusion": grouped["path_c_confusion_bool"].sum(),
            "score_rate": grouped["score_value"].mean(),
            "path_c_rate": grouped["path_c_confusion_bool"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)
    summary["path_c_pct"] = (summary["path_c_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])
    return summary


score_summary_table("variant", order=VARIANT_ORDER)

,n,score_true,score_false,unparseable,path_c_confusion,score_rate,path_c_rate,score_pct,path_c_pct
variant,,,,,,,,,
open_probs,54,4,46,4,31,0.074074,0.574074,7.4,57.4
mc_numeric_probs,54,26,10,18,2,0.481481,0.037037,48.1,3.7


In [3]:
score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)

,n,score_true,score_false,unparseable,path_c_confusion,score_rate,path_c_rate,score_pct,path_c_pct
vignette_name,,,,,,,,,
CA Trump voter,12,5,5,2,2,0.416667,0.166667,41.7,16.7
college STEM work,12,2,5,5,3,0.166667,0.250000,16.7,25.0
covid vaccine (blue/red),12,4,4,4,4,0.333333,0.333333,33.3,33.3
diabetes insulin obese,12,3,7,2,5,0.250000,0.416667,25.0,41.7
discharged weapon (last year),12,5,5,2,4,0.416667,0.333333,41.7,33.3
english teacher humanities,12,0,10,2,4,0.000000,0.333333,0.0,33.3
healthcare employment,12,5,6,1,5,0.416667,0.416667,41.7,41.7
military overseas (federal pool),12,2,8,2,3,0.166667,0.250000,16.7,25.0
professional drivers speeding,12,4,6,2,3,0.333333,0.250000,33.3,25.0


## `mc_numeric_probs` detail

MC options, parsed letter, normative letter, score, and whether the choice is the **P(T|C)** lure.

In [4]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]
MC_LURE_COLS = [f"option_{letter}_lure" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, label_col, lure_col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS, MC_LURE_COLS):
        label = row.get(label_col)
        lure = row.get(lure_col)
        if pd.notna(label) and str(label).strip():
            lure_text = f" [{lure}]" if pd.notna(lure) and str(lure).strip() else ""
            parts.append(f"{letter}: {label}{lure_text}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "normative_choice",
        "p_t_given_c",
        "score",
        "score_value",
        "path_c_confusion",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 160)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,normative_choice,p_t_given_c,score,score_value,path_c_confusion,answer_line
0,CA Trump voter,A: About 42% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,,A,0.31,false,0,false,- P(Trump and Southern CA voter) = 0.078 × 0.27 = 0.02
1,CA Trump voter,A: About 42% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
2,CA Trump voter,A: About 42% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,,A,0.31,false,0,false,P(Voted Trump | CA voter
3,CA Trump voter,A: About 42% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
4,CA Trump voter,A: About 42% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
5,CA Trump voter,A: About 42% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,A,A,0.31,true,1,false,A
12,college STEM work,A: About 24% [Bayes P(C|T)] | B: About 63% [P(T|C)*P(T|D) confusion] | C: About 74% [P(T|D) confusion] | D: About 17% [P(D) confusion] | E: About 85% [P(T|C...,,A,0.85,false,0,false,"First, I need to find P(return). Without complete information about the overlap between STEM and employment, and return"
13,college STEM work,A: About 24% [Bayes P(C|T)] | B: About 63% [P(T|C)*P(T|D) confusion] | C: About 74% [P(T|D) confusion] | D: About 17% [P(D) confusion] | E: About 85% [P(T|C...,,A,0.85,false,0,false,"For P(returned), I need overall return rate. The problem doesn't directly give this, but the only relevant numbers I can use suggest using the"
14,college STEM work,A: About 24% [Bayes P(C|T)] | B: About 63% [P(T|C)*P(T|D) confusion] | C: About 74% [P(T|D) confusion] | D: About 17% [P(D) confusion] | E: About 85% [P(T|C...,,A,0.85,false,0,false,"First, I need to find P(returned). However, I'm not given enough information to calculate this exactly since I"
15,college STEM work,A: About 24% [Bayes P(C|T)] | B: About 63% [P(T|C)*P(T|D) confusion] | C: About 74% [P(T|D) confusion] | D: About 17% [P(D) confusion] | E: About 85% [P(T|C...,A,A,0.85,true,1,false,A


## `open_probs` detail

Re-parse open responses and compare to normative **P(C|T)** and **P(T|C)**.

In [11]:
from benchmarks.base_rate import matches_scepticism_target, parse_open_response
from benchmarks.simple_rate import PATH_C_LURE_NAME, load_benchmark, matches_path_c_confusion

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    return pd.Series(
        {
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.answer_type != "unparseable",
            "score_true": matches_scepticism_target(item, parsed),
            "path_c_confusion_rescored": matches_path_c_confusion(item, parsed),
            "p_t_given_c_pct": float(row["p_t_given_c"]) * 100,
        }
    )


def _csv_bool(series: pd.Series) -> pd.Series:
    return series.astype(bool).map(lambda value: "true" if value else "false")


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = open_probs.drop(columns=["score_true"], errors="ignore")
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

idx = open_probs.index
rescored_percent = pd.to_numeric(open_probs["parsed_percent_rescored"], errors="coerce")
df.loc[idx, "parsed_percent"] = rescored_percent.map(
    lambda value: "" if pd.isna(value) else f"{value:g}"
)
df.loc[idx, "score_true"] = open_probs["score_true"].astype(bool)
df.loc[idx, "score_value"] = open_probs["score_true"].astype(int)
if "score" in df.columns:
    df.loc[idx, "score"] = _csv_bool(open_probs["score_true"])
df.loc[idx, "path_c_confusion_bool"] = open_probs["path_c_confusion_rescored"].astype(bool)
if "path_c_confusion" in df.columns:
    df.loc[idx, "path_c_confusion"] = _csv_bool(open_probs["path_c_confusion_rescored"])

open_probs_view = open_probs[
    [
        "example_id",
        "vignette_name",
        "normative_percent",
        "normative_open",
        "p_t_given_c_pct",
        "parsed_numbers",
        "parsed_percent_rescored",
        "score_true",
        "path_c_confusion_rescored",
    ]
].sort_values("vignette_name")

print(
    "Rescored open_probs normative pass:",
    int(open_probs["score_true"].sum()),
    "/",
    len(open_probs),
    "| P(T|C) confusion:",
    int(open_probs["path_c_confusion_rescored"].sum()),
    "/",
    len(open_probs),
)
open_probs_view

TypeError: Invalid value for dtype 'str'. Value should be a string or missing value (or array of those).

## `open_probs` vs `mc_numeric_probs` vs normative

Side-by-side for all 10 vignettes. **Normative** = P(C|T) (`normative_percent`). **P(T|C)** = `p_t_given_c` (inverse-conditional lure).

In [ ]:
from benchmarks.base_rate import parse_open_response, parse_response
from benchmarks.simple_rate import PATH_C_LURE_NAME
from scripts.build_base_rate_prompts import _load_overlap

OVERLAP_VIGNETTE_NAMES = {v.name for v in _load_overlap()}

items_meta = pd.read_csv(ROOT / "data" / "simple" / "items.csv")


def _label_percent(label: str) -> float | None:
    text = (label or "").strip()
    if not text.startswith("About "):
        return None
    try:
        return float(text.removeprefix("About ").removesuffix("%"))
    except ValueError:
        return None


def _format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter in "ABCDE":
        label = row.get(f"option_{letter.lower()}_label")
        if pd.notna(label) and str(label).strip():
            parts.append(f"{letter}: {label}")
    return " | ".join(parts)


comparison_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = item_mc.get(f"option_{mc_choice.lower()}_label", "") if mc_choice else ""
    mc_lure = item_mc.get(f"option_{mc_choice.lower()}_lure", "") if mc_choice else ""
    mc_pct = _label_percent(str(mc_label))

    normative_pct = float(item_open["normative_percent"])
    path_c_pct = float(item_open["p_t_given_c"]) * 100
    open_pct = parsed_open.percent

    comparison_rows.append(
        {
            "vignette_name": vignette_name,
            "overlap": vignette_name in OVERLAP_VIGNETTE_NAMES,
            "normative_open": item_open["normative_open"],
            "normative_pct": normative_pct,
            "p_t_given_c_pct": path_c_pct,
            "open_parsed_pct": open_pct,
            "open_delta_vs_norm_pp": None if open_pct is None else open_pct - normative_pct,
            "open_score": bool(open_row.get("score_true", open_row.get("score_value", 0))),
            "open_path_c": bool(open_row.get("path_c_confusion_bool", False)),
            "mc_choices": _format_mc_choices(item_mc),
            "mc_choice": mc_choice,
            "mc_label": mc_label,
            "mc_lure": mc_lure,
            "mc_parsed_pct": mc_pct,
            "mc_delta_vs_norm_pp": None if mc_pct is None else mc_pct - normative_pct,
            "mc_score": bool(mc_row.get("score_true", mc_row.get("score_value", 0))),
            "mc_path_c": mc_lure == PATH_C_LURE_NAME or (
                mc_pct is not None and abs(mc_pct - path_c_pct) <= 0.5
            ),
            "normative_mc_letter": item_mc["normative_choice"],
        }
    )

open_vs_mc = pd.DataFrame(comparison_rows).sort_values("vignette_name")

print(
    "open_probs pass:",
    int(open_vs_mc["open_score"].sum()),
    "/",
    len(open_vs_mc),
    "| mc_numeric_probs pass:",
    int(open_vs_mc["mc_score"].sum()),
    "/",
    len(open_vs_mc),
    "| open P(T|C) confusion:",
    int(open_vs_mc["open_path_c"].sum()),
    "| mc P(T|C) confusion:",
    int(open_vs_mc["mc_path_c"].sum()),
)

COMPARISON_COLUMNS = [
    "vignette_name",
    "overlap",
    "normative_pct",
    "open_parsed_pct",
    "open_score",
    "mc_choices",
    "mc_choice",
    "mc_label",
    "mc_score",
]

pd.set_option("display.max_colwidth", 160)
display(open_vs_mc[COMPARISON_COLUMNS])

open_probs pass: 0 / 10 | mc_numeric_probs pass: 5 / 10 | open P(T|C) confusion: 5 | mc P(T|C) confusion: 1


,vignette_name,overlap,normative_pct,open_parsed_pct,open_score,mc_choices,mc_choice,mc_label,mc_score
0,CA Trump voter,False,42.100,4.9,False,A: About 42% | B: About 8% | C: About 31% | D: About 27% | E: About 5%,,,False
1,actor waiter overlap,True,12.190,0.0,False,A: About 12% | B: About 55% | C: About 36% | D: About 0% | E: About 65%,D,About 0%,False
2,college STEM work,True,23.850,85.0,False,A: About 24% | B: About 63% | C: About 74% | D: About 17% | E: About 85%,E,About 85%,False
3,covid vaccine (blue/red),False,66.200,10.0,False,A: About 66% | B: About 1% | C: About 8% | D: About 10% | E: About 19%,A,About 66%,True
4,diabetes insulin obese,True,42.810,20.0,False,A: About 43% | B: About 5% | C: About 16% | D: About 20% | E: About 3%,A,About 43%,True
5,discharged weapon (last year),False,77.270,13.0,False,A: About 77% | B: About 0% | C: About 13% | D: About 30%,A,About 77%,True
6,english teacher humanities,True,52.060,60.0,False,A: About 52% | B: About 69% | C: About 55% | D: About 0% | E: About 38%,D,About 0%,False
7,healthcare employment,False,9.091,60.0,False,A: About 9% | B: About 10% | C: About 32% | D: About 60% | E: About 54%,H,NaN,False
8,military overseas (federal pool),False,38.350,50.0,False,A: About 38% | B: About 37% | C: About 58% | D: About 64% | E: About 20%,A,About 38%,True
9,professional drivers speeding,True,85.490,35.0,False,A: About 85% | B: About 0% | C: About 16% | D: About 10% | E: About 2%,A,About 85%,True


In [ ]:
basically, whenever overlap is false, everything is correct; but when overlap is true, the open output is wrong, but the mc is correct.
COuld this be due to the MC being way to easy?

SyntaxError: invalid syntax (1242660209.py, line 1)

### Printable comparison

Per vignette: source probabilities from `items.csv` (P(C), P(D), P(T|C), P(T|D)), normative / open / MC answers, and full `mc_numeric_probs` prompt.

In [ ]:
import re

from benchmarks.base_rate import parse_open_response, parse_response

benchmark_df = pd.read_csv(ROOT / "data" / "simple" / "benchmark.csv")


def mc_numeric_options_prompt(prompt: str) -> str:
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    option_lines = [line for line in lines if re.match(r"^[A-E]\.\s", line)]
    return " | ".join(option_lines)


def format_source_ps(item_row: pd.Series) -> str:
    return " | ".join(
        [
            f"P(C)={float(item_row['p_c']):.6g}",
            f"P(D)={float(item_row['p_d']):.6g}",
            f"P(T|C)={float(item_row['p_t_given_c']):.6g}",
            f"P(T|D)={float(item_row['p_t_given_d']):.6g}",
        ]
    )


print_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]
    bench_row = benchmark_df.loc[benchmark_df["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = (
        str(item_mc.get(f"option_{mc_choice.lower()}_label", ""))
        if mc_choice
        else ""
    )
    full_prompt = str(bench_row["prompt"])

    print_rows.append(
        {
            "vignette_name": vignette_name,
            "source_ps": format_source_ps(item_open),
            "normative_pct": float(item_open["normative_percent"]),
            "p_t_given_c_pct": float(item_open["p_t_given_c"]) * 100,
            "open_parsed_pct": parsed_open.percent,
            "mc_label": f"{mc_choice} {mc_label}".strip(),
            "numeric_prompt": mc_numeric_options_prompt(full_prompt),
            "prompt": full_prompt,
        }
    )

print_table = pd.DataFrame(print_rows).sort_values("vignette_name")

print(f"{'vignette_name':<32} {'normative':>10} {'P(T|C)':>10} {'open':>10} {'MC label':>14}")
print("-" * 84)
for row in print_table.itertuples(index=False):
    open_pct = "—" if pd.isna(row.open_parsed_pct) else f"{row.open_parsed_pct:.4g}%"
    print(f"\n{row.vignette_name}")
    print(f"  source Ps:   {row.source_ps}")
    print(f"  normative:   {row.normative_pct:.4g}%  (P(C|T))")
    print(f"  P(T|C):      {row.p_t_given_c_pct:.4g}%  (inverse-conditional lure)")
    print(f"  open parsed: {open_pct}")
    print(f"  MC label:    {row.mc_label}")
    print(f"  numeric prompt: {row.numeric_prompt}")
    print("  prompt:")
    for line in row.prompt.splitlines():
        print(f"    {line}")

print_table.drop(columns=["prompt"])

## Optional: split by model when multiple LLMs are present

In [ ]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["path_c_confusion_bool"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")